# Primary MERFISH ALDEx input preparation and result postprocessing

This short notebook:

1. Loads the finalized primary MERFISH AnnData object.
2. Exports the two count-matrix/metadata input groups used across the primary-dataset ALDEx runs.
3. Postprocesses four distinct sets of ALDEx outputs documented in `ALDEx runs.xlsx`.

The runs are intentionally kept separate because they differ in model specification, anatomical subset, significance filtering, and downstream use.

## 1. Imports and paths

In [ ]:
from __future__ import annotations

from glob import glob
from pathlib import Path

import scanpy as sc

import kimlabspatial.differential_expression as de

from scale_aware_st import (
    RepositoryConfig,
    load_pooled_aldex_spec,
)


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
config = RepositoryConfig.from_env()
ALDEX_RESULT_MODE = "published"  # choose "published" or "recompute"
PROJECT_DIR = config.root
DEG_DIR = config.results_dir / "DEG"
DATA_PATH = config.data_dir / "primary_merfish" / "analysis_objects" / "adata_glia_aldex_pp.h5ad"

ALL_INPUT_DIR = DEG_DIR / "all"
CT_REGION_INPUT_DIR = DEG_DIR / "ct_region"

GENE_INFO_PATH = config.resources_dir / "primary_merfish" / "MERFISH_Master-List-03_SM.xlsx"

RUN_DATE = "07142025"

for directory in (DEG_DIR, ALL_INPUT_DIR, CT_REGION_INPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if ALDEX_RESULT_MODE not in {"published", "recompute"}:
    raise ValueError("ALDEX_RESULT_MODE must be 'published' or 'recompute'.")

if ALDEX_RESULT_MODE == "published":
    required_mode_inputs = [config.additional_data_dir / "Additional_File_4.xlsx"]
    mode_message = "Published mode: using frozen pooled primary ALDEx results."
else:
    required_mode_inputs = [DATA_PATH, GENE_INFO_PATH]
    mode_message = (
        "Recompute mode is staged: this notebook exports ALDEx inputs, "
        "the separate R workflow fits the models, and this notebook then "
        "postprocesses the individual workbooks. See RECOMPUTE_WORKFLOW.md."
    )
missing_mode_inputs = [path for path in required_mode_inputs if not path.is_file()]
if missing_mode_inputs:
    raise FileNotFoundError(
        f"{ALDEX_RESULT_MODE.title()} mode is missing required input(s): "
        + ", ".join(map(str, missing_mode_inputs))
    )
print(mode_message)

## 2. Load the primary MERFISH AnnData

In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    adata = sc.read_h5ad(DATA_PATH)
    print(adata)
else:
    adata = None
    print("Published mode: skipping the primary AnnData load and ALDEx input generation.")


## 3. Prepare ALDEx inputs

Two preprocessing groups are exported:

- all cells grouped by cell type;
- cell type × region strata using `obs["ct_region"]`.

These inputs are reused by multiple downstream ALDEx model specifications.

In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    de.prep_for_aldex2(
        adata,
        str(ALL_INPUT_DIR),
        "cell_type_all",
        RUN_DATE,
    )


In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    de.prep_for_aldex2(
        adata,
        str(CT_REGION_INPUT_DIR),
        "cell_type",
        RUN_DATE,
        obs_key="ct_region",
    )


## 4. Postprocess ALDEx results

### 4.1 Cell type × region informed-TSS results

In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    aldex_ct_region_files = sorted(
        glob(
            str(
                DEG_DIR
                / "cell_type_clusters_aldex3_informedtss_results"
                / "aldex_mem_cell_type_*.xlsx"
            )
        )
    )

    if not aldex_ct_region_files:
        raise FileNotFoundError(
            "No per-stratum primary cell-type x region ALDEx workbooks were found. "
            "The input-export cells do not fit ALDEx; run the corresponding R workflow, "
            "place its workbooks under results/DEG/cell_type_clusters_aldex3_informedtss_results/, "
            "then rerun this postprocessing cell. See RECOMPUTE_WORKFLOW.md."
        )

    aldex_ct_region_prefix = str(
        DEG_DIR
        / "cell_type_clusters_aldex3_informedtss_results"
        / "aldex_mem_cell_type_"
    )

    de.postprocess_aldex2(
        aldex_results=aldex_ct_region_files,
        curr_dir=str(DEG_DIR),
        s1=aldex_ct_region_prefix,
        s2=f"_results_{RUN_DATE}",
        savestr="aldex_ct_inftss_results",
        exp_col="age_binary",
        savestr2="aldex_ct_inftss_pivot_signed.xlsx",
        gene_info=str(GENE_INFO_PATH),
        signed=True,
    )
elif ALDEX_RESULT_MODE == "published":
    aldex_ct_region_results = load_pooled_aldex_spec(config.additional_data_dir, "primary_celltype_region")
    print(f"Loaded {len(aldex_ct_region_results)} pooled cell type × region strata.")
else:
    raise ValueError("ALDEX_RESULT_MODE must be published or recompute")


### 4.2 All-cell-type informed-TSS results — significant only

In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    aldex_all_ct_files = sorted(
        glob(
            str(
                DEG_DIR
                / "cell_type_clusters_Aldex3_ALL_it_results"
                / "aldex_mem_cell_type_blmm_all_*.xlsx"
            )
        )
    )

    if not aldex_all_ct_files:
        raise FileNotFoundError(
            "No per-stratum primary all-region cell-type ALDEx workbooks were found. "
            "Run the corresponding R workflow and place its workbooks under "
            "results/DEG/cell_type_clusters_Aldex3_ALL_it_results/ before continuing. "
            "See RECOMPUTE_WORKFLOW.md."
        )

    aldex_all_ct_prefix = str(
        DEG_DIR
        / "cell_type_clusters_Aldex3_ALL_it_results"
        / "aldex_mem_cell_type_blmm_all_"
    )

    de.postprocess_aldex2(
        aldex_results=aldex_all_ct_files,
        curr_dir=str(DEG_DIR),
        s1=aldex_all_ct_prefix,
        s2=f"_results_{RUN_DATE}",
        significant=True,
        savestr="aldex_main_ds_all_SIG_ct_results",
        exp_col="age_binary",
        savestr2="aldex_main_ds_all_pivot.xlsx",
        pivot=True,
        gene_info=False,
        signed=True,
    )
elif ALDEX_RESULT_MODE == "published":
    aldex_all_ct_results = load_pooled_aldex_spec(config.additional_data_dir, "primary_celltype")
    print(f"Loaded {len(aldex_all_ct_results)} pooled cell-type strata.")
else:
    raise ValueError("ALDEX_RESULT_MODE must be published or recompute")


### 4.3 Anterior cell type × region results — all genes

In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    aldex_anterior_files = sorted(
        glob(
            str(
                DEG_DIR
                / "cell_type_clusters_aldex3_anterior_it_results"
                / "aldex_mem_cell_type_anterior*.xlsx"
            )
        )
    )

    if not aldex_anterior_files:
        raise FileNotFoundError(
            "No per-stratum anterior primary ALDEx workbooks were found. "
            "Run the corresponding R workflow and place its workbooks under "
            "results/DEG/cell_type_clusters_aldex3_anterior_it_results/ before continuing. "
            "See RECOMPUTE_WORKFLOW.md."
        )

    aldex_anterior_prefix = str(
        DEG_DIR
        / "cell_type_clusters_aldex3_anterior_it_results"
        / "aldex_mem_cell_type_anterior"
    )

    de.postprocess_aldex2(
        aldex_results=aldex_anterior_files,
        curr_dir=str(DEG_DIR),
        s1=aldex_anterior_prefix,
        s2=f"_results_{RUN_DATE}",
        significant=False,
        savestr="aldex_ALL_ct_anterior_inftss_results",
        exp_col="age_binary",
        savestr2="",
        pivot=False,
        gene_info=str(GENE_INFO_PATH),
        signed=True,
    )
elif ALDEX_RESULT_MODE == "published":
    aldex_anterior_results = load_pooled_aldex_spec(config.additional_data_dir, "primary_anterior_celltype_region")
    print(f"Loaded {len(aldex_anterior_results)} pooled anterior strata.")
else:
    raise ValueError("ALDEX_RESULT_MODE must be published or recompute")


### 4.4 Cell type × region BLMM results — all genes

In [ ]:
if ALDEX_RESULT_MODE == "recompute":
    aldex_blmm_prefix = str(
        DEG_DIR
        / "cell_type_clusters_aldex3_mem_blmm_results"
        / "aldex_mem_cell_type_blmm_"
    )

    aldex_blmm_files = sorted(
        glob(f"{aldex_blmm_prefix}*.xlsx")
    )

    if not aldex_blmm_files:
        raise FileNotFoundError(
            "No per-stratum BLMM sensitivity workbooks were found. Full BLMM outputs "
            "are not redistributed; recomputation requires the pinned development engine "
            "documented in the ALDEx workflow README."
        )

    de.postprocess_aldex2(
        aldex_results=aldex_blmm_files,
        curr_dir=str(DEG_DIR),
        s1=aldex_blmm_prefix,
        s2=f"_results_{RUN_DATE}",
        significant=False,
        savestr="aldex_ALL_ct_it_blmm_results",
        exp_col="age_binary",
        savestr2="",
        pivot=False,
    )
elif ALDEX_RESULT_MODE == "published":
    print("Full per-stratum BLMM sensitivity outputs are not redistributed; use Additional File 4, sheet ALDEx_lme_vs_blmm_engine, for the published engine comparison.")
else:
    raise ValueError("ALDEX_RESULT_MODE must be published or recompute")
